# MVROC quickstart

Figures for the catalogue README. This notebook is the place for the period comparison and for further diagrams.

Run it from the repository root. It reads `mvroc.csv` and writes `gallery/mvroc-sbdb-periods.png`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = Path.cwd()
if not (root / "mvroc.csv").is_file():
    root = root.parent

catalogue = pd.read_csv(root / "mvroc.csv")
catalogue.head()

## Periods against the literature

`p_rot_h` is the adopted MVROC period. `rot_per_sbdb` is the JPL Small-Body Database period. `rot_per_lsm_gst` is the Lomb–Scargle period of Greenstreet et al. (2026). Several solutions of one asteroid share the published abscissa.

In [ ]:
sbdb = catalogue.dropna(subset=["p_rot_h", "rot_per_sbdb"])
greenstreet = catalogue.dropna(subset=["p_rot_h", "rot_per_lsm_gst"])

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
panels = (
    (axes[0], greenstreet, "rot_per_lsm_gst", "o", "Greenstreet et al. (2026)"),
    (axes[1], sbdb, "rot_per_sbdb", "s", "SBDB"),
)
for ax, part, column, marker, title in panels:
    x = part[column].to_numpy(float)
    y = part["p_rot_h"].to_numpy(float)
    ax.scatter(x, y, s=18, marker=marker, c="0.2", zorder=3)
    n_obj = part["provid"].nunique()
    ax.set_title(f"{title} ({n_obj} objects, {len(part)} solutions)")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Published period [h]")
    ax.grid(True, which="both", alpha=0.3)
    ax.set_aspect("equal", adjustable="box")

periods = np.concatenate([
    sbdb["rot_per_sbdb"], sbdb["p_rot_h"],
    greenstreet["rot_per_lsm_gst"], greenstreet["p_rot_h"],
])
limits = np.array([periods.min() * 0.7, periods.max() * 1.4])
for ax in axes:
    ax.plot(limits, limits, "k-", lw=1, label="1:1")
    ax.plot(limits, 2 * limits, "k--", lw=1, label="2:1")
    ax.plot(limits, 0.5 * limits, "k:", lw=1, label="1:2")
    ax.set_xlim(limits)
    ax.set_ylim(limits)
axes[0].set_ylabel("MVROC period [h]")
axes[0].legend(loc="upper left")
fig.tight_layout()

out = root / "gallery" / "mvroc-sbdb-periods.png"
out.parent.mkdir(exist_ok=True)
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"wrote {out}")
print(
    f"Greenstreet: {greenstreet['provid'].nunique()} objects, {len(greenstreet)} solutions; "
    f"SBDB: {sbdb['provid'].nunique()} objects, {len(sbdb)} solutions"
)